# Computational Derivations and Spectral Invariants of Girard Torsion, Jordan Algebra H3(O), and 3-Torus Eigenmodes Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.k1n2-dzgt/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s in the dataset. For Croissant datasets, record sets and fields are accessed via their `@id`.

In [ ]:
# Access the available record sets
record_sets = metadata.recordSet  # List of record set metadata objects

if not record_sets:
    print("No record sets found in this dataset. Check schema or data availability.")
else:
    for rs in record_sets:
        print(f"RecordSet @id: {rs['@id']}")
        if 'field' in rs:
            print("  Fields:")
            for field in rs['field']:
                print(f"    - Field @id: {field['@id']} (name: {field.get('name','')})")
        if 'column' in rs:
            print("  Columns:")
            for col in rs['column']:
                print(f"    - Column @id: {col['@id']} (name: {col.get('name','')})")


## 3. Data Extraction
Load data from each record set into a DataFrame for analysis and map each with its `@id`. Column and field `@id`s from the previous overview are used.

In [ ]:
# Extract data from all record sets into DataFrames
dataframes = {}
record_set_ids = []

# Collect all record set @ids
if metadata.recordSet:
    for rs in metadata.recordSet:
        record_set_ids.append(rs['@id'])

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded record set: {record_set_id}, columns: {df.columns.tolist()}")
        print(df.head())
    except Exception as e:
        print(f"Could not load records for {record_set_id}: {e}")

# If at least one record set, inspect its columns
if record_set_ids:
    first_rs = record_set_ids[0]
    print(f"Columns for record set {first_rs}:")
    print(dataframes[first_rs].columns.tolist())
    display(dataframes[first_rs].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. All column references use their `@id`.

In [ ]:
# Choose the first available record set and inspect its fields/columns
if record_set_ids:
    rs_id = record_set_ids[0]
    df = dataframes[rs_id]
    
    # If schema provides numeric fields, select one by @id (example, you may update these below)
    numeric_fields = [col for col in df.columns if 'torsion' in col or 'eigenvalue' in col or 'spectral' in col or 'value' in col or df[col].dtype in ['float64','int64']]
    if numeric_fields:
        numeric_field_id = numeric_fields[0]
        print(f"Selected numeric field: {numeric_field_id}")
        threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 10
        # Safe threshold: median or 10
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}")
        print(filtered_df.head())
        
        # Normalization
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
        
        # Try grouping by another field (pick one with categorical data)
        group_fields = [col for col in df.columns if 'parameter' in col or 'class' in col or 'category' in col or df[col].dtype == 'object']
        group_field_id = group_fields[0] if group_fields else None
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"Grouped data by {group_field_id}:")
            print(grouped_df.head())
    else:
        print("No numeric fields found for EDA.")
else:
    print("No record sets to analyze.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. All references are by column `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize numeric distribution from EDA section
if record_set_ids and 'numeric_field_id' in locals():
    plt.figure(figsize=(6,4))
    sns.histplot(df[numeric_field_id], kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()
    
    # If normalized column exists
    if f"{numeric_field_id}_normalized" in filtered_df.columns:
        plt.figure(figsize=(6,4))
        sns.histplot(filtered_df[f"{numeric_field_id}_normalized"], kde=True)
        plt.title(f'Normalized Distribution of {numeric_field_id}')
        plt.xlabel(f"{numeric_field_id}_normalized")
        plt.ylabel('Frequency')
        plt.show()
    
    # If grouped_df exists and group_field_id
    if 'grouped_df' in locals() and group_field_id:
        grouped_df[numeric_field_id].plot(kind='bar')
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.xlabel(group_field_id)
        plt.tight_layout()
        plt.show()
else:
    print("Visualization skipped: No numeric field or data available.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- This notebook demonstrates loading, overview, extraction, and initial EDA on the FAIR^2 dataset using `mlcroissant`.
- All data elements are referenced by their `@id`, ensuring reproducibility and clarity for each entity.
- Further analysis can leverage the full Croissant schema to select additional record sets or perform deeper statistical or topological investigation based on available fields.
- For broader use, consult the schema (via the Croissant URL) to understand field semantics and provenance details.